# STA 2503 · Project 1 — Notebook 01: Part I, scaled random walks → Brownian motion

**How this one works (lighter than the warm-up):** the Python plumbing (plots, array shapes, saving figures) is already written and commented. You only fill the `____` blanks, and each blank is a **maths → code** step taken from Lecture 3. Every blank has a hint right next to it. ⏱ roughly 30–45 min.

| Project task | Where |
|---|---|
| 1. Definition of Brownian motion | Section 0 (for the report — no code) |
| 2. Simulate BM, $N_1 = 10{,}000$ | Section A |
| 3. Scaled random walk, $N_2 = 10$ | Section B |
| 4–5. Repeat for $N_2 = 50, 200, 1000$ and compare with BM | Section C |
| 6. Discuss (with numbers to back it up) | Sections D–E |

Figures are saved straight into `report/figures/`, ready for `\includegraphics` in `main.tex`.

---
## 0. Definition (goes in the report, from Lecture 3)
A process $W = \{W_t\}_{t \ge 0}$ on $(\Omega, \mathcal F, \{\mathcal F_t\}_{t\ge0}, \mathbb P)$ is a **standard Brownian motion** if
1. $W_0 = 0$ and $W$ is adapted to $\{\mathcal F_t\}$;
2. for $0 \le s < t$, the increment $W_t - W_s \sim \mathcal N(0,\, t-s)$;
3. $W_t - W_s$ is independent of $\mathcal F_s$;
4. $t \mapsto W_t(\omega)$ is continuous for every $\omega$.

**The two facts the code uses:** on a grid with spacing $\Delta t$, each increment is an independent $\mathcal N(0, \Delta t)$ draw, i.e. **standard deviation $\sqrt{\Delta t}$**; and the path is the running sum of those increments, starting at 0.

In [ ]:
# SETUP — run as is.
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm          # normal distribution: norm.pdf(x) is the N(0,1) density
from pathlib import Path

FIG_DIR = Path("report/figures")      # relative to this notebook's folder (project1/)
FIG_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(2503)     # ONE random generator for the whole notebook -> reproducible report

def check(condition, hint):
    if condition:
        print("✅ correct")
    else:
        raise AssertionError("❌ not yet — hint: " + hint)

---
## A. Simulating Brownian motion (task 2)

Recipe for `n_paths` paths on $[0,1]$ with `N` steps:
1. grid spacing $\Delta t$ and time grid $0, \Delta t, 2\Delta t, \dots, 1$ ($N+1$ points);
2. draw a `(n_paths, N)` array of increments $\Delta W \sim \mathcal N(0, \Delta t)$;
3. running sum along each row (`axis=1`), then put a column of zeros in front so every path starts at $W_0 = 0$.

Python notes: `rng.normal(mean, sd, size=...)` wants the **standard deviation**, like R's `rnorm`. The function receives `rng` as an argument so all randomness comes from the one generator created above.

In [ ]:
def simulate_bm(N, n_paths, rng):
    '''Brownian motion on [0, 1] with N steps.
    Returns t (shape N+1) and W (shape n_paths x N+1); each row of W is one path.'''
    dt = ____                                          # hint: [0,1] cut into N equal pieces
    t  = ____                       # hint: same as warm-up 3.1 — N steps need N+1 points
    dW = rng.normal(0, ____, size=(n_paths, N))  # hint: sd, not variance!  W_{t+dt} - W_t ~ N(0, dt)

    W = np.cumsum(dW, axis=1)                             # running sum along each path (row)
    W = np.hstack([np.zeros((n_paths, 1)), W])            # prepend W_0 = 0  -> shape (n_paths, N+1)
    return t, W

In [ ]:
# CHECK A — uses a small N and many paths so it's fast. Theory: W_1 ~ N(0, 1).
t_chk, W_chk = simulate_bm(N=500, n_paths=20_000, rng=np.random.default_rng(0))
W1_chk = W_chk[:, -1]                    # [:, -1] = every row (path), last column (t = 1)
print(f"mean of W_1 = {W1_chk.mean():.3f} (theory 0),  var of W_1 = {W1_chk.var():.3f} (theory 1)")
check(W_chk.shape == (20_000, 501) and np.all(W_chk[:, 0] == 0) and np.isclose(t_chk[1], 1/500)
      and abs(W1_chk.var() - 1) < 0.05,
      "If var(W_1) is ~0.002 you passed the VARIANCE dt as the sd; it must be np.sqrt(dt). "
      "If the shape is wrong, check t has N+1 points.")

In [ ]:
# Task 2: five BM paths with N1 = 10,000 — run as is.
N1 = 10_000
t_bm, W_bm = simulate_bm(N1, n_paths=5, rng=rng)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_bm, W_bm.T, linewidth=0.8)        # .T: plot draws one line per COLUMN, so transpose -> one line per path
ax.axhline(0, color="black", linewidth=0.5)
ax.set_xlabel("$t$", fontsize=13); ax.set_ylabel("$W_t$", fontsize=13)
ax.set_title(f"Brownian motion, 5 sample paths ($N_1 = {N1:,}$ steps)", fontsize=13)
fig.savefig(FIG_DIR / "part1_bm_paths.pdf", bbox_inches="tight")
plt.show()

---
## B. The scaled symmetric random walk (task 3)

From Lecture 3: $Z_i \overset{iid}{\sim} \pm 1$ with probability $\tfrac12$, and
$$W^{(N)}_t = \frac{1}{\sqrt N} \sum_{i=1}^{Nt} Z_i, \qquad t \in \{0, \tfrac1N, \dots, 1\}.$$
Same recipe as BM, except the increments are $\pm \frac{1}{\sqrt N}$ coin flips instead of normal draws. Note both have mean 0 and variance $\frac1N = \Delta t$ — that's the whole reason the limit works.

Python note: `rng.choice([a, b], size=...)` picks from the list with equal probabilities (R: `sample(c(a, b), n, replace = TRUE)`).

In [ ]:
def scaled_random_walk(N, n_paths, rng):
    '''Scaled symmetric random walk W^(N) on [0, 1].
    Returns t (shape N+1) and W (shape n_paths x N+1).'''
    t = np.linspace(0, 1, N + 1)
    Z = rng.choice(____, size=(n_paths, N))       # hint: the two possible coin-flip values
    X = np.cumsum(Z, axis=1)                           # unscaled walk X_n = Z_1 + ... + Z_n
    X = np.hstack([np.zeros((n_paths, 1)), X])         # X_0 = 0
    W = ____                               # hint: the 1/sqrt(N) scaling in the formula above
    return t, W

In [ ]:
# CHECK B
t_rw, W_rw = scaled_random_walk(10, n_paths=20_000, rng=np.random.default_rng(1))
steps = np.diff(W_rw, axis=1)            # the increments W_{t+dt} - W_t
print("distinct step sizes:", np.unique(np.round(steps, 6)), "   var of W_1:", round(W_rw[:, -1].var(), 3))
check(W_rw.shape == (20_000, 11) and np.allclose(np.abs(steps), 1/np.sqrt(10)) and abs(W_rw[:, -1].var() - 1) < 0.05,
      "Each step should be ±1/sqrt(10) ≈ ±0.316. Z takes the values -1 and 1; W = X divided by sqrt(N).")

In [ ]:
# Task 3: five scaled-random-walk paths with N2 = 10 — run as is.
t10, W10 = scaled_random_walk(10, n_paths=5, rng=rng)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t10, W10.T, marker="o", markersize=3)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_xlabel("$t$", fontsize=13); ax.set_ylabel("$W^{(N)}_t$", fontsize=13)
ax.set_title("Scaled symmetric random walk, 5 sample paths ($N_2 = 10$)", fontsize=13)
fig.savefig(FIG_DIR / "part1_rw_N10.pdf", bbox_inches="tight")
plt.show()

---
## C. Increasing $N_2$ and comparing with BM (tasks 4–5)

One figure, four panels ($N_2 = 10, 50, 200, 1000$). Each panel shows 5 random-walk paths, with the BM paths from Section A in light grey behind them for reference, on the same axes so they're comparable.

Python notes (all pre-written):
- `fig, axes = plt.subplots(2, 2)` makes a 2×2 grid; `axes.flat` lists the 4 panels in order.
- `zip(a, b)` walks through two lists side by side: first panel with first `N`, and so on.

In [ ]:
N2_list = [10, 50, 200, 1000]

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
for ax, N2 in zip(axes.flat, N2_list):
    t_N, W_N = ____    # hint: call your Section B function with this panel's N2

    ax.plot(t_bm, W_bm.T, color="lightgrey", linewidth=0.6)    # BM reference (from Section A)
    ax.plot(t_N, W_N.T, linewidth=1)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.set_title(f"$N_2 = {N2}$", fontsize=13)

for ax in axes[1, :]: ax.set_xlabel("$t$", fontsize=13)
for ax in axes[:, 0]: ax.set_ylabel("$W^{(N_2)}_t$", fontsize=13)
fig.suptitle("Scaled random walks (colour) vs Brownian motion (grey)", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "part1_rw_vs_bm.pdf", bbox_inches="tight")
plt.show()

---
## D. Backing the discussion with numbers (task 6)

A picture of 5 paths is suggestive; these two checks make the discussion rigorous and tie it to Lecture 3.

**D1 — Distribution of $W^{(N)}_1$ (CLT).** $W^{(N)}_1 = \frac{1}{\sqrt N}\sum_{i=1}^N Z_i \to \mathcal N(0,1)$. Simulate 20,000 paths for each $N_2$, take the value at $t=1$, and compare its histogram with the $\mathcal N(0,1)$ density.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2), sharey=True)
x = np.linspace(-4, 4, 400)
summary = []                                              # we'll collect a row of numbers per N2 for a table

for ax, N2 in zip(axes, N2_list):
    t_N, W_N = scaled_random_walk(N2, n_paths=20_000, rng=rng)
    W1 = ____                                      # hint: all paths, value at the LAST time point (t = 1)

    # W1 can only take values (2k - N2)/sqrt(N2): a lattice with spacing 2/sqrt(N2).
    # Centre one histogram bin on each lattice point so bar heights estimate the density fairly.
    edges = np.arange(-N2 - 1, N2 + 2, 2) / np.sqrt(N2)
    edges = edges[np.abs(edges) <= 4.5]
    ax.hist(W1, bins=edges, density=True, alpha=0.6, label="simulated")
    ax.plot(x, ____, color="black", label="N(0,1)")    # hint: the N(0,1) density evaluated on x (see setup cell)
    ax.set_title(f"$N_2 = {N2}$", fontsize=12); ax.set_xlabel("$W^{(N_2)}_1$")

    increments = np.diff(W_N, axis=1)
    qv = np.sum(____, axis=1)                  # hint: Lecture 3, QV = sum of SQUARED increments (per path)
    summary.append((N2, W1.mean(), W1.var(), qv.mean()))

axes[0].set_ylabel("density"); axes[0].legend(fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / "part1_W1_histograms.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# CHECK D1
check(len(summary) == 4 and all(abs(v - 1) < 0.05 for _, _, v, _ in summary)
      and all(np.isclose(q, 1) for *_, q in summary),
      "W1 = last column of W_N. QV: square the increments (** 2) before summing along axis=1.")

**D2 — Summary table.** Mean and variance of $W^{(N)}_1$, and the quadratic variation over $[0,1]$, next to the BM path from Section A. Run as is; the printed table is what you'd copy into the report (as a LaTeX `tabular`).

In [ ]:
qv_bm = np.sum(np.diff(W_bm, axis=1) ** 2, axis=1)       # QV of each BM path from Section A

print(f"{'model':>16} | {'mean W_1':>9} | {'var W_1':>8} | {'QV on [0,1]':>11}")
print("-" * 54)
for N2, m, v, q in summary:
    print(f"{'RW, N2 = ' + str(N2):>16} | {m:9.3f} | {v:8.3f} | {q:11.4f}")
print(f"{'BM, N1 = 10,000':>16} | {'(theory 0)':>9} | {'(theory 1)':>8} | {qv_bm.mean():11.4f}")

---
## E. What to write in the discussion (task 6)

Look at your figures and table, then answer these in a few sentences each in the report — they're the observations the marker is looking for:

1. **Shape of the paths.** How does the walk look for $N_2 = 10$ versus $N_2 = 1000$? Each step is $\pm 1/\sqrt{N_2}$ over time $1/N_2$, so the slope of each segment is $\pm\sqrt{N_2}$. What happens to the slopes as $N_2 \to \infty$, and what does that say about differentiability of the limit (BM)?
2. **Distribution at a fixed time.** From D1: for small $N_2$ the histogram is lumpy (a lattice); for large $N_2$ it matches $\mathcal N(0,1)$. Which theorem is this? (Lecture 3: CLT applied to $W^{(N)}_t - W^{(N)}_s$.)
3. **Which BM properties already hold for every $N_2$, and which only in the limit?** Think: $W_0 = 0$, mean 0, variance $t$, independent increments, martingale, normal increments, continuous paths.
4. **Quadratic variation.** Why is the random walk's QV *exactly* 1 for every path, while BM's is only ≈ 1? (Hint: $(\pm 1/\sqrt{N})^2 = 1/N$, summed $N$ times.) Link to $[W, W]_t = t$ from Lecture 3.
5. **The big picture.** Donsker's theorem: the whole *path* converges in distribution to BM, not just the value at one time. Why does this matter for Part II? (The binomial tree is a random walk in $\log S_t$; its limit is GBM, which is why the CRR price converges to Black–Scholes.)

When you're done: `git add .` → `git commit -m "Part I: random walks and BM"` → `git push`.